In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("all_matches.csv")
df.head()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Scotland,False


In [3]:
pd.unique(df[['home_team','away_team']].values.ravel())

array(['Scotland', 'England', 'Wales', 'Ireland', 'Uruguay', 'Argentina',
       'Austria', 'Hungary', 'Bohemia', 'Belgium', 'France',
       'Switzerland', 'Netherlands', 'British Guiana',
       'Trinidad and Tobago', 'South Africa', 'Germany', 'Sweden',
       'Norway', 'Denmark', 'Italy', 'Chile', 'Finland', 'Luxembourg',
       'Russia', 'Philippines', 'China', 'Brazil', 'Suriname',
       'United States', 'Japan', 'Paraguay', 'Egypt', 'Greece', 'Spain',
       'Czechoslovakia', 'Yugoslavia', 'Estonia', 'Northern Ireland',
       'Costa Rica', 'El Salvador', 'Guatemala', 'Honduras', 'Poland',
       'Portugal', 'Romania', 'New Zealand', 'Australia', 'Latvia',
       'Mexico', 'Lithuania', 'Turkey', 'Aruba', 'Curaçao', 'Bulgaria',
       'Canada', 'Soviet Union', 'Haiti', 'Jamaica', 'Kenya', 'Uganda',
       'Bolivia', 'Azerbaijan', 'Armenia', 'Georgia', 'Peru',
       'British Honduras', 'Dutch East Indies', 'Barbados', 'Nicaragua',
       'Cuba', 'Faroe Islands', 'Iceland', 'Mart

In [5]:
df['weight'] = df['tournament'].apply(get_tournament_weight)
df.groupby('tournament')['weight'].first().sort_values(ascending=False).head(30)

tournament
World Cup                        4.0
Copa América                     3.0
Confederations Cup               3.0
Asian Cup                        3.0
European Championship            3.0
CONCACAF Championship            3.0
Olympic Games                    3.0
Copa America                     3.0
African Nations Cup              3.0
European Nations League B        2.5
European Nations League A/B      2.5
CONCACAF Nations League B        2.5
European Nations League          2.5
Copa America qualifier           2.5
CONCACAF Nations League C        2.5
European Nations League B/C      2.5
European Nations League C        2.5
CONCACAF Nations League q        2.5
CONCACAF Nations League A        2.5
Copa América qualifier           2.5
CONCACAF Nations League          2.5
European Nations League A        2.5
Asian Cup qualifier              2.5
WC and Oce Cup q                 2.5
World Cup and Asian Cup qual     2.5
World Cup qualifier              2.5
World Cup q & Nordic Ch    

In [6]:
from Confederations import get_confederation

all_teams = pd.unique(df[['home_team', 'away_team']].values.ravel())
unknowns = [t for t in all_teams if get_confederation(t) == "Unknown"]
print(sorted(unknowns))

[]


In [ ]:
def get_tournament_weight(tournament,home_team,away_team):
    t=tournament.strip()
    home_conf = get_confederation(home_team)
    away_conf = get_confederation(away_team)
    
    conf_priority = ["UEFA", "CONMEBOL", "CONCACAF", "CAF", "AFC", "OFC", "Unknown"]
    home_rank = conf_priority.index(home_conf)
    away_rank = conf_priority.index(away_conf)
    top_conf = home_conf if home_rank <= away_rank else away_conf
    if t == "World Cup":
        return 4.0
    
    major_championships = [
        "European Championship",
        "Copa America", "Copa América",
        "African Nations Cup",
        "Asian Cup",
        "Confederations Cup",
        "Olympic Games",
        "CONCACAF Championship"
    ]
    if t in major_championships:
        return 3.0
    
    major_qualifiers=[
        "World Cup qualifier",
        "European Championship qual",
        "African Nations Cup qualifier",
        "Asian Cup qualifier",
        "Copa America qualifier", "Copa América qualifier",
        "CONCACAF Ch q", "World Cup q & Nordic Ch",
        "NA Champ & WC qual", "World Cup q & British Ch",
        "World Cup and CONCACAF Ch q", "World Cup and Asian Cup qual",
        "World Cup and African Cup qual", "WC and Oce Cup q",
        "WC q and Oce Cup"
    ]
    major_qualifier_keywords=[
        "European Nations League",
        "CONCACAF Nations League"
    ]
    if t in major_qualifiers:
        return 2.5
    if any(kw in t for kw in major_qualifier_keywords):
        return 2.5
    minor_keywords = [
        "qualifier", "Championship", "Cup qualifier",
        "Games qualifier", "CONCACAF", "Gulf Cup",
        "Caribbean Cup", "CECAFA", "COSAFA",
        "Southeast Asian Champ", "South Asian Championship",
        "West Asian Championship", "East Asian Championship",
        "Central American Cup", "UNCAF", "CFU Championship",
        "Oceania Nations Cup", "Copa Centenario"
    ]
    if any(kw.lower() in t.lower() for kw in minor_keywords):
        return 2.0
    regional_keywords = [
        "Cup", "Tournament", "Games", "Championship",
        "Friendly tournament", "Mundialito", "Bolivarian",
        "Panamerican", "Mediterranean", "Nordic", "Baltic",
        "Balkan", "Arab Cup", "Gulf", "Kings Cup", "King's Cup"
    ]
    if any(kw.lower() in t.lower() for kw in regional_keywords):
        return 1.5
    return 1.0